# Application d'Analyse de Sentiments avec Gradio

Parmi les modèles proposés (TextBlob Pattern, TextBlob Naive Bayes, VADER, Flair), nous avons choisi **deux modèles** :

1. **TextBlob** — un modèle basé sur un lexique de règles, rapide et léger.
2. **VADER** — un modèle basé sur un lexique spécialement conçu pour les textes courts (réseaux sociaux, avis...).



## 1. Installation des bibliothèques

In [14]:
# On installe Gradio ainsi que les deux modèles d'analyse de sentiments
%pip install gradio textblob vaderSentiment


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Définition des fonctions



In [15]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# On initialise une seule fois l'analyseur VADER (réutilisé à chaque appel)
analyseur_vader = SentimentIntensityAnalyzer()


def analyser_avec_textblob(texte):
    """Analyse le sentiment d'un texte avec TextBlob (Pattern).

    La polarité va de -1 (très négatif) à +1 (très positif).
    """
    polarite = TextBlob(texte).sentiment.polarity

    if polarite > 0:
        etiquette = "Positif"
    elif polarite < 0:
        etiquette = "Négatif"
    else:
        etiquette = "Neutre"

    return f"{etiquette}  (polarité = {polarite:.2f})"


def analyser_avec_vader(texte):
    """Analyse le sentiment d'un texte avec VADER.

    Le score 'compound' va de -1 (très négatif) à +1 (très positif).
    """
    scores = analyseur_vader.polarity_scores(texte)
    compound = scores["compound"]

    if compound >= 0.05:
        etiquette = "Positif"
    elif compound <= -0.05:
        etiquette = "Négatif"
    else:
        etiquette = "Neutre"

    return f"{etiquette}  (compound = {compound:.2f})"

In [16]:
# Fonction principale appelée par Gradio :
# elle choisit le bon modèle selon le menu déroulant.
def analyser_sentiment(texte, modele):
    if not texte or texte.strip() == "":
        return "Veuillez saisir un texte à analyser."

    if modele == "TextBlob (Pattern)":
        return analyser_avec_textblob(texte)
    elif modele == "VADER":
        return analyser_avec_vader(texte)
    else:
        return "Modèle inconnu."

## 3. Test rapide des fonctions

In [17]:
exemple = "I really love this product, it is amazing!"
print("TextBlob :", analyser_sentiment(exemple, "TextBlob (Pattern)"))
print("VADER    :", analyser_sentiment(exemple, "VADER"))

TextBlob : Positif  (polarité = 0.62)
VADER    : Positif  (compound = 0.86)


## 4. Construction de l'interface Gradio (2 onglets)

In [18]:
import gradio as gr

with gr.Blocks() as app:
    # Titre et description de l'application
    gr.Markdown("####### Application d'Analyse de Sentiments ######")
    gr.Markdown(
        "Cette application analyse le **sentiment** d'un texte (positif, négatif ou neutre) "
        "à l'aide de deux modèles au choix : **TextBlob (Pattern)** et **VADER**."
    )

    # Onglets : Démo et Documentation
    with gr.Tabs():

        # ----- Onglet 1 : Démo -----
        with gr.Tab("Démo"):
            texte_entree = gr.Textbox(
                label="Texte à analyser",
                placeholder="Saisissez votre texte ici...",
                lines=4,
            )
            choix_modele = gr.Radio(
                choices=["TextBlob (Pattern)", "VADER"],
                value="TextBlob (Pattern)",
                label="Choisissez le modèle",
            )
            resultat = gr.Textbox(label="Résultat de l'analyse", interactive=False)

            with gr.Row():
                bouton_analyser = gr.Button("Analyser")
                bouton_effacer = gr.Button("Effacer")

            # Exemples cliquables
            gr.Examples(
                examples=[
                    ["I really love this product, it is amazing!", "TextBlob (Pattern)"],
                    ["This is the worst experience I have ever had.", "VADER"],
                    ["The meeting is scheduled for tomorrow at noon.", "VADER"],
                ],
                inputs=[texte_entree, choix_modele],
                label="Essayez ces exemples !",
            )

            # Actions des boutons
            bouton_analyser.click(
                fn=analyser_sentiment,
                inputs=[texte_entree, choix_modele],
                outputs=resultat,
            )
            bouton_effacer.click(
                lambda: ("", ""),
                inputs=None,
                outputs=[texte_entree, resultat],
            )

        # ----- Onglet 2 : Documentation -----
        with gr.Tab("Documentation"):
            gr.Markdown("## Comment ça marche ?")
            gr.Markdown(
                "Cette application utilise deux modèles d'**analyse de sentiments** pour déterminer "
                "si un texte exprime une opinion **positive**, **négative** ou **neutre**."
            )
            gr.Markdown("### Les deux modèles")
            gr.Markdown(
                "- **TextBlob (Pattern)** : modèle basé sur un lexique de règles. Il calcule une "
                "**polarité** comprise entre -1 (très négatif) et +1 (très positif). Simple et rapide.\n"
                "- **VADER** (*Valence Aware Dictionary and sEntiment Reasoner*) : modèle basé sur un "
                "lexique optimisé pour les textes courts (réseaux sociaux, avis). Il renvoie un score "
                "**compound** entre -1 et +1."
            )
            gr.Markdown("### Règles d'interprétation")
            gr.Markdown(
                "- **Positif** : score > 0 (TextBlob) ou compound ≥ 0.05 (VADER)\n"
                "- **Négatif** : score < 0 (TextBlob) ou compound ≤ -0.05 (VADER)\n"
                "- **Neutre** : sinon"
            )
            gr.Markdown("### Fonctionnalités")
            gr.Markdown(
                "- **Choix du modèle** : sélectionnez TextBlob ou VADER avant l'analyse.\n"
                "- **Bouton Analyser** : lance l'analyse du texte saisi.\n"
                "- **Bouton Effacer** : vide les champs de saisie et de résultat.\n"
                "- **Exemples** : testez l'application avec des phrases prédéfinies."
            )
            gr.Markdown(
                "Remarque : ces deux modèles sont optimisés pour l'**anglais**. "
                "Les résultats sur des textes en français peuvent être moins fiables."
            )

# Lancement de l'interface
app.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
